# Auto Loader — Live Demo: Continuous Ingestion from ADLS
### GlobalMart Data Engineering · College Presentation Notebook (v2)

---

## What We Are Proving

> **Auto Loader does NOT re-read files it has already processed.**  
> **When a NEW file lands in ADLS, Auto Loader detects it automatically — zero manual intervention.**

---

## Live Demo Run Order

| # | Cell | Action | Expected Result |
|---|------|--------|------------------|
| 1 | Setup | Configure ADLS connection | All paths printed |
| 2 | Clean Start | Wipe Bronze, checkpoint, schema | Fresh state |
| 3 | Phase 1 | Load all files with `availableNow=True` | ~1.4M rows in Bronze, stream stops |
| 4 | Phase 2 | Write `orders_new.csv` to staging | File ready, NOT in `raw/` |
| 5 | Phase 3 | Start continuous stream (same checkpoint) | Stream idle — no new files |
| 6 | **Phase 4** | Copy `orders_new.csv` into `raw/` | **Stream detects it, count jumps +5** |
| 7 | Validation | No-duplicate proof, checkpoint inspection | Exactly-once confirmed |

---

## Auto Loader vs Manual `spark.read`

| Feature | `spark.read` (manual) | Auto Loader (`cloudFiles`) |
|---------|----------------------|----------------------------|
| Tracks processed files | No — re-reads everything | Yes — checkpoint remembers every file |
| New file detection | Must retrigger entire job | Automatic on next interval |
| Duplicate risk on re-run | High | Zero |
| Production grade | No | Yes |

---
## How Auto Loader Tracks Files — The Checkpoint

```
ADLS raw/ folder:
  orders.csv          ← Phase 1: processed ✅  (checkpoint remembers this)
  orders_new.csv      ← Phase 4: we drop this  🔴 (not yet seen)

Checkpoint folder (_checkpoints/orders_demo_v2/):
  sources/   ← records EVERY processed file (path + size + mtime)
  offsets/   ← current stream reading position
  commits/   ← confirmed completed micro-batch IDs

What happens every trigger (30 seconds):
  1. List all files in raw/
  2. Compare against checkpoint sources/
  3. orders.csv      → already in checkpoint → SKIP
  4. orders_new.csv  → NOT in checkpoint     → PROCESS  ← this is the moment
  5. Write 5 new rows to Bronze Delta
  6. Update checkpoint — orders_new.csv now recorded

Next trigger: both files in checkpoint → nothing to do → stream idles
```

> **Each file is processed exactly once — this is exactly-once delivery.**

In [ ]:
# ─── SETUP — Run this cell FIRST before anything else ─────────────────────────
# WARNING: Do NOT push this notebook to GitHub with the real key

storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← replace before running

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

base_path       = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
raw_path        = f"{base_path}/raw"
bronze_path     = f"{base_path}/bronze"
staging_path    = f"{base_path}/staging"
checkpoint_path = f"{base_path}/_checkpoints/orders_demo_v2"
output_path     = f"{bronze_path}/orders_demo_v2"
schema_path     = f"{base_path}/_autoloader/schemas/orders_demo_v2"

print("Setup complete!")
print(f"  raw_path        : {raw_path}")
print(f"  output_path     : {output_path}")
print(f"  checkpoint_path : {checkpoint_path}")
print(f"  staging_path    : {staging_path}")

In [ ]:
# ─── Check what files are currently in raw/ ────────────────────────────────────
print("Files in ADLS raw/ folder (Auto Loader will process ALL of these in Phase 1):")
print("-" * 65)

csv_files = [f for f in dbutils.fs.ls(raw_path) if f.name.endswith(".csv")]
for f in csv_files:
    print(f"  {f.name:<35} {f.size / 1024 / 1024:>8.2f} MB")

print(f"\nTotal CSV files: {len(csv_files)}")
print("\nNOTE: orders_new.csv should NOT be here yet. If it is, remove it before Clean Start.")
if any(f.name == "orders_new.csv" for f in csv_files):
    print("  ⚠ orders_new.csv found in raw/ — removing it now...")
    dbutils.fs.rm(f"{raw_path}/orders_new.csv")
    print("  Removed.")

In [ ]:
# ─── CLEAN START ───────────────────────────────────────────────────────────────
# Deletes Bronze table, checkpoint, and Auto Loader schema cache.
# Safe to run even on first run — silently skips if nothing exists yet.

for path, label in [
    (output_path,     "Bronze output table (orders_demo_v2)"),
    (checkpoint_path, "Checkpoint folder   (_checkpoints/orders_demo_v2)"),
    (schema_path,     "Auto Loader schema  (_autoloader/schemas/orders_demo_v2)"),
]:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"  Deleted : {label}")
    except:
        print(f"  Skipped : {label} (did not exist)")

print()
print("Clean start ready — run Phase 1 next.")

In [ ]:
# ─── PHASE 1: Load all current files with availableNow=True ───────────────────
# Column renaming cleans special characters before writing to Delta:
#   "Cost (Rs)" → "Cost_Rs"  (parentheses and spaces removed)
# This avoids DELTA_INVALID_CHARACTERS errors without needing column mapping mode.

from pyspark.sql.functions import current_timestamp, input_file_name

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",      "true")
    .option("inferSchema", "true")
    .load(raw_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

for c in raw_stream.columns:
    clean = (
        c.replace(" ", "_")
         .replace("(", "")
         .replace(")", "")
         .replace(".", "")
         .replace("%", "Pct")
         .replace("-", "_")
    )
    if clean != c:
        raw_stream = raw_stream.withColumnRenamed(c, clean)

phase1_query = (
    raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema",        "true")
    .trigger(availableNow=True)
    .start(output_path)
)

phase1_query.awaitTermination()
print("Phase 1 complete — all current files processed, stream stopped.")

In [ ]:
# ─── Phase 1 Results ──────────────────────────────────────────────────────────
from pyspark.sql.functions import count, min as spark_min, max as spark_max

bronze_df    = spark.read.format("delta").load(output_path)
phase1_count = bronze_df.count()

print("=" * 60)
print("  PHASE 1 RESULTS")
print("=" * 60)
print(f"  Total rows in Bronze : {phase1_count:,}")
print()
print("Rows per source file:")
(
    bronze_df
    .groupBy("_source_file")
    .agg(
        count("*").alias("row_count"),
        spark_min("_ingested_at").alias("first_ingested"),
        spark_max("_ingested_at").alias("last_ingested")
    )
    .show(truncate=False)
)
print(f"Baseline saved: phase1_count = {phase1_count:,}")
print("This number must NOT change until we drop orders_new.csv in Phase 4.")

In [ ]:
# ─── Sample orders rows with metadata columns ──────────────────────────────────
from pyspark.sql.functions import col

print("Sample rows from orders.csv in Bronze:")
print()
(
    spark.read.format("delta").load(output_path)
    .filter(col("_source_file").contains("orders.csv"))
    .select("OrderID", "CustomerID", "OrderChannel", "OrderDate", "_ingested_at", "_source_file")
    .limit(5)
    .show(truncate=False)
)
print("After Phase 4, rows from orders_new.csv will appear with a LATER _ingested_at.")

---
## Phase 2 — Prepare the New File (Staging)

We create `orders_new.csv` with 5 fresh GlobalMart orders and write it to **staging** using `dbutils.fs.put()`.
It is **not** in `raw/` yet — Auto Loader cannot see it.

> Staging = holding area. We control exactly when Auto Loader sees the file by copying it into `raw/`.

In [ ]:
# ─── PHASE 2: Write orders_new.csv to staging ─────────────────────────────────
# dbutils.fs.put() writes directly to ADLS — no Azure Portal upload needed.

orders_new_content = """\
OrderID,CustomerID,OrderDate,ShippingDate,ExpectedDeliveryDate,ActualDeliveryDate,ShippingTierID,SupplierID,OrderChannel
ORD-NEW-2026-001,CUST-08966,2026-06-15,2026-06-16,2026-06-20,,SHP-001,SUP-01,Online
ORD-NEW-2026-002,CUST-14521,2026-06-15,2026-06-16,2026-06-21,,SHP-002,SUP-02,Mobile App
ORD-NEW-2026-003,CUST-29834,2026-06-15,2026-06-17,2026-06-22,,SHP-001,SUP-03,Online
ORD-NEW-2026-004,CUST-07312,2026-06-15,2026-06-16,2026-06-20,,SHP-003,SUP-01,In-Store
ORD-NEW-2026-005,CUST-35671,2026-06-15,2026-06-18,2026-06-23,,SHP-002,SUP-04,Mobile App"""

staged_csv = f"{staging_path}/orders_new.csv"
dbutils.fs.put(staged_csv, orders_new_content, overwrite=True)

print(f"File written to staging: {staged_csv}")
print()
preview_df = spark.read.option("header", "true").csv(staged_csv)
print("Preview:")
preview_df.show(truncate=False)
print(f"Row count : {preview_df.count()}")
print()
print("File is in staging — NOT yet visible to Auto Loader (watching raw/ only).")

In [ ]:
# ─── Confirm file locations before Phase 3 ────────────────────────────────────
raw_csv_names     = [f.name for f in dbutils.fs.ls(raw_path)     if f.name.endswith(".csv")]
staging_csv_names = [f.name for f in dbutils.fs.ls(staging_path) if f.name.endswith(".csv")]

print("raw/ folder (Auto Loader watches this):")
for n in raw_csv_names:
    print(f"  {n}")
print(f"  orders_new.csv present in raw/     : {'orders_new.csv' in raw_csv_names}")

print()
print("staging/ folder (holding area):")
for n in staging_csv_names:
    print(f"  {n}")
print(f"  orders_new.csv present in staging/ : {'orders_new.csv' in staging_csv_names}")

print()
print("CONFIRMED: orders_new.csv is in staging only. Auto Loader is unaware of it.")

---
## Phase 3 — Start Continuous Stream

We start Auto Loader with `trigger(processingTime='30 seconds')`.  
It uses the **same checkpoint** as Phase 1 — so it already knows all current files were processed.

```
Phase 1 checkpoint state:  orders.csv + all other CSVs → DONE
Phase 3 first trigger:     all files → SKIP (in checkpoint), nothing new → IDLE
Phase 4 (file dropped):    orders_new.csv → NEW → PROCESS
```

> The stream runs **in the background** — you can run subsequent cells while it is active.

In [ ]:
# ─── PHASE 3: Start continuous stream (same checkpoint as Phase 1) ─────────────
# IMPORTANT: Column renaming must be identical to Phase 1.
# Same renaming loop → same column names → no schema mismatch on the Bronze table.

from pyspark.sql.functions import current_timestamp, input_file_name

continuous_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",      "true")
    .option("inferSchema", "true")
    .load(raw_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

for c in continuous_stream.columns:
    clean = (
        c.replace(" ", "_")
         .replace("(", "")
         .replace(")", "")
         .replace(".", "")
         .replace("%", "Pct")
         .replace("-", "_")
    )
    if clean != c:
        continuous_stream = continuous_stream.withColumnRenamed(c, clean)

continuous_query = (
    continuous_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema",        "true")
    .trigger(processingTime="30 seconds")
    .start(output_path)
)

print("Continuous stream started!")
print(f"  Stream ID  : {continuous_query.id}")
print(f"  Is active  : {continuous_query.isActive}")
print(f"  Trigger    : every 30 seconds")
print(f"  Watching   : {raw_path}")
print(f"  Output     : {output_path}")
print()
print("Stream runs in the background. Continue to the next cell.")

In [ ]:
# ─── Confirm: count unchanged (no new files yet) ───────────────────────────────
import time
time.sleep(5)  # let stream settle

current_count = spark.read.format("delta").load(output_path).count()

print("Stream status BEFORE dropping new file:")
print(f"  Phase 1 baseline : {phase1_count:,}")
print(f"  Current count    : {current_count:,}")
print(f"  New rows added   : {current_count - phase1_count:,}")
print()
print("Expected: 0 new rows — stream is idle, all files already in checkpoint.")

---
## Phase 4 — THE DEMO MOMENT

### Drop `orders_new.csv` into `raw/`

```
BEFORE:  raw/ has only original files  → stream idle → count unchanged

ACTION:  Copy orders_new.csv from staging/ into raw/
         Next trigger fires within 30 seconds
         Auto Loader detects new file → ingests 5 rows

AFTER:   count = baseline + 5
         _source_file = orders_new.csv for those 5 rows
         _ingested_at = LATER timestamp than Phase 1 rows
```

**Run the drop cell, then immediately run the watcher cell.**

In [ ]:
# ─── DROP THE FILE — this is the demo moment ──────────────────────────────────
print("=" * 65)
print("  DROPPING orders_new.csv INTO ADLS raw/ FOLDER")
print("=" * 65)
print()

dbutils.fs.cp(
    f"{staging_path}/orders_new.csv",
    f"{raw_path}/orders_new.csv"
)

print(f"File dropped: {raw_path}/orders_new.csv")
print()
print("Files now in raw/:")
for f in dbutils.fs.ls(raw_path):
    if f.name.endswith(".csv"):
        print(f"  {f.name}")
print()
print("Auto Loader trigger fires every 30 seconds.")
print("Run the next cell NOW to watch the row count update.")

In [ ]:
# ─── WATCH the row count update in real time ──────────────────────────────────
import time

print(f"Baseline (Phase 1): {phase1_count:,} rows")
print("Checking every 15 seconds for up to 3 minutes...")
print()

detected = False
for i in range(12):
    count_now = spark.read.format("delta").load(output_path).count()
    new_rows  = count_now - phase1_count
    elapsed   = i * 15
    status    = "NEW FILE DETECTED!" if new_rows > 0 else "waiting..."

    print(f"  [{elapsed:>3}s]  Total: {count_now:>12,}  |  New rows: {new_rows:>5}  |  {status}")

    if new_rows > 0 and not detected:
        detected = True
        print()
        print("=" * 65)
        print(f"  AUTO LOADER DETECTED AND INGESTED orders_new.csv")
        print(f"  {new_rows} new rows added automatically")
        print("=" * 65)
        break

    time.sleep(15)

if not detected:
    print()
    print("Not detected within 3 minutes.")
    print(f"Check stream is running: continuous_query.isActive = {continuous_query.isActive}")

In [ ]:
# ─── Show rows by source file after Phase 4 ───────────────────────────────────
from pyspark.sql.functions import count, min as spark_min, max as spark_max

print("Bronze table breakdown by source file:")
(
    spark.read.format("delta").load(output_path)
    .groupBy("_source_file")
    .agg(
        count("*").alias("row_count"),
        spark_min("_ingested_at").alias("ingested_from"),
        spark_max("_ingested_at").alias("ingested_to")
    )
    .orderBy("ingested_from")
    .show(truncate=False)
)
print("The _ingested_at timestamps prove orders_new.csv was ingested LATER than orders.csv.")

In [ ]:
# ─── Show the 5 new rows specifically ─────────────────────────────────────────
from pyspark.sql.functions import col

print("The 5 new orders ingested automatically from orders_new.csv:")
(
    spark.read.format("delta").load(output_path)
    .filter(col("_source_file").contains("orders_new"))
    .select("OrderID", "CustomerID", "OrderChannel", "OrderDate", "_ingested_at", "_source_file")
    .show(truncate=False)
)
print("These rows did not exist in Bronze before Phase 4.")
print("No manual trigger — Auto Loader detected and ingested them automatically.")

---
## Validation — Proving Correctness

| Claim | Cell | Expected Result |
|-------|------|-----------------|
| No duplicates | v2-21 | `COUNT(*) == COUNT(DISTINCT OrderID)` |
| orders.csv not re-processed | v2-22 | Ingest timestamps all from Phase 1 |
| Checkpoint records processed files | v2-23 | `sources/` folder lists all files |
| Delta history shows two writes | v2-24 | Version 0 = Phase 1, last version = Phase 4 |

In [ ]:
# ─── VALIDATION 1: Zero Duplicates ────────────────────────────────────────────
bronze_df       = spark.read.format("delta").load(output_path)
total_rows      = bronze_df.count()
distinct_orders = bronze_df.select("OrderID").distinct().count()
duplicates      = total_rows - distinct_orders

print("DUPLICATE CHECK:")
print(f"  Total rows in Bronze     : {total_rows:,}")
print(f"  Distinct OrderIDs        : {distinct_orders:,}")
print(f"  Duplicates               : {duplicates:,}")
print()

if duplicates == 0:
    print("PASSED: Zero duplicates.")
    print("Auto Loader processed each file exactly once despite two stream runs.")
else:
    print(f"WARNING: {duplicates:,} duplicates found — investigate checkpoint.")

In [ ]:
# ─── VALIDATION 2: orders.csv was NOT re-processed in Phase 3/4 ───────────────
from pyspark.sql.functions import col, min as spark_min, max as spark_max

historical_window = (
    spark.read.format("delta").load(output_path)
    .filter(col("_source_file").contains("orders.csv"))
    .filter(~col("_source_file").contains("orders_new"))
    .agg(
        spark_min("_ingested_at").alias("earliest"),
        spark_max("_ingested_at").alias("latest")
    )
    .first()
)

new_file_window = (
    spark.read.format("delta").load(output_path)
    .filter(col("_source_file").contains("orders_new"))
    .agg(
        spark_min("_ingested_at").alias("earliest"),
        spark_max("_ingested_at").alias("latest")
    )
    .first()
)

print("INGEST TIMELINE:")
print(f"  orders.csv     : {historical_window['earliest']}  →  {historical_window['latest']}")
print(f"  orders_new.csv : {new_file_window['earliest']}  →  {new_file_window['latest']}")
print()
print("orders.csv timestamps are all from Phase 1.")
print("orders_new.csv timestamps are from Phase 4 — LATER.")
print("This proves orders.csv was processed exactly once — checkpoint worked.")

In [ ]:
# ─── VALIDATION 3: Checkpoint Folder Inspection ────────────────────────────────
def describe_checkpoint_item(name):
    d = {
        "metadata":  "stream query ID + configuration",
        "sources/":  "LIST of every file already processed",
        "offsets/":  "current reading position in the stream",
        "commits/":  "confirmed completed micro-batch IDs",
        "state/":    "aggregation state (GROUP BY queries only)",
    }
    return d.get(name, "checkpoint metadata")

print(f"CHECKPOINT FOLDER: {checkpoint_path}")
print()
for item in dbutils.fs.ls(checkpoint_path):
    print(f"  {item.name:<20} ← {describe_checkpoint_item(item.name)}")

print()
print("The 'sources/' folder is the key to exactly-once processing.")
print("It lists every file path Auto Loader has already ingested.")
print("Any file found here will NEVER be re-processed.")

In [ ]:
# ─── VALIDATION 4: Delta Table Version History ─────────────────────────────────
print("DELTA WRITE HISTORY (one version per micro-batch write):")
(
    spark.sql(f"DESCRIBE HISTORY delta.`{output_path}`")
    .select("version", "timestamp", "operation", "operationMetrics")
    .show(10, truncate=False)
)
print("Version 0 = Phase 1 (initial load of all existing files)")
print("Last version = Phase 4 (orders_new.csv detected and ingested)")

In [ ]:
# ─── Stop the stream + Final Summary ──────────────────────────────────────────
if continuous_query.isActive:
    continuous_query.stop()
    print("Continuous stream stopped.")
else:
    print("Stream already stopped.")

final_df    = spark.read.format("delta").load(output_path)
final_count = final_df.count()
final_files = final_df.select("_source_file").distinct().count()
dupes       = final_count - final_df.select("OrderID").distinct().count()

print()
print("=" * 60)
print("  DEMO COMPLETE — FINAL SUMMARY")
print("=" * 60)
print(f"  Total rows in Bronze     : {final_count:,}")
print(f"  Distinct source files    : {final_files}")
print(f"  Duplicate rows           : {dupes}")
print()
print("What was proved:")
print("  1. All existing files ingested in Phase 1 — checkpoint recorded them")
print("  2. Continuous stream started — idle (all files already in checkpoint)")
print("  3. orders_new.csv dropped into raw/ — detected within 30 seconds")
print("  4. 5 new rows added — existing files were NOT re-processed")
print("  5. Zero duplicates — exactly-once processing confirmed")

---
## Viva Q&A

### Q: What is the main advantage of Auto Loader over `spark.read`?
**A:** `spark.read` has no memory — it re-reads ALL files on every run, causing duplicates.  
Auto Loader uses a checkpoint to record every processed file. On the next run it only reads NEW files — cheaper, faster, no duplicates.

---

### Q: What is in the checkpoint folder? Where is it stored?
**A:** `sources/` (list of all processed files), `offsets/` (read position), `commits/` (completed batch IDs).  
Stored in ADLS: `abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/_checkpoints/orders_demo_v2`

---

### Q: How does Auto Loader detect new files?
**A:** Every trigger interval it lists all files in the watched folder.  
It compares this list against `sources/` in the checkpoint.  
Files not in `sources/` → process. Files already in `sources/` → skip.

---

### Q: What is the difference between `availableNow=True` and `processingTime`?
**A:** `availableNow=True` → one-shot batch: process all pending files, then **stop**.  
`processingTime='30 seconds'` → continuous: check every 30 sec, keep running indefinitely.

---

### Q: How do we prove no duplicates?
**A:** `COUNT(*) == COUNT(DISTINCT OrderID)`. If equal → zero duplicates. See cell v2-21.

---

### Q: What does `_source_file` tell us?
**A:** Added by `input_file_name()` during ingestion. It stores the full ADLS path of the source file.  
Lets us trace any Bronze row back to the exact file it came from — critical for auditing.

---

### Q: What is `schemaEvolutionMode = addNewColumns`?
**A:** If a new file arrives with an extra column, Auto Loader adds it to the schema automatically.  
Existing rows get `null` for that column. Without this mode, a schema mismatch would fail the stream.

---

### Q: Why rename columns before writing to Delta?
**A:** Delta rejects column names with special characters like spaces, parentheses, or dots.  
`shipping_tier.csv` has `Cost (Rs)` — renaming it to `Cost_Rs` before the write avoids the `DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES` error.  
Both Phase 1 and Phase 3 must use the **same renaming logic** so the Bronze table schema stays consistent.

---

## Architecture — End to End

```
ADLS raw/
  orders.csv + 9 other CSVs  → Phase 1: ingested (checkpoint records all)
  orders_new.csv             → Phase 4: detected + ingested automatically
        |
        ▼  cloudFiles (Auto Loader)
  Schema stored  → _autoloader/schemas/orders_demo_v2/
  Files tracked  → _checkpoints/orders_demo_v2/sources/
        |
        ▼  Column renaming (Cost (Rs) → Cost_Rs)
        |
        ▼  writeStream → Delta
  bronze/orders_demo_v2/
    All rows from all files
    _source_file  = which file this row came from
    _ingested_at  = when Auto Loader ingested it
```